In [1]:
# Импортируем необходимые библиотеки
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy as sp
import operator
import warnings

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
from surprise import SVD, Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import accuracy

In [2]:
warnings.filterwarnings('ignore')

# Загрузка
df = pd.read_parquet('source/steam_200k.parquet')
df.drop_duplicates(inplace=True)

# Размер датасета
print("Размер датасета:", df.shape)

# Первые строки
df.head()


Размер датасета: (199293, 4)


,user_id,game_name,status,played_hour
0,151603712,The Elder Scrolls V Skyrim,purchase,1.0
1,151603712,The Elder Scrolls V Skyrim,play,273.0
2,151603712,Fallout 4,purchase,1.0
3,151603712,Fallout 4,play,87.0
4,151603712,Spore,purchase,1.0


In [3]:
# Оставляем только записи с меткой 'play' и временем >= 2 часов
df = df[(df['status'] == 'play') & (df['played_hour'] >= 2)]
df

,user_id,game_name,status,played_hour
1,151603712,The Elder Scrolls V Skyrim,play,273.0
3,151603712,Fallout 4,play,87.0
5,151603712,Spore,play,14.9
7,151603712,Fallout New Vegas,play,12.1
9,151603712,Left 4 Dead 2,play,8.9
...,...,...,...,...
199985,128470551,Nether,play,2.8
199987,128470551,Rogue Legacy,play,2.6
199989,128470551,Mortal Kombat Komplete Edition,play,2.5
199991,128470551,Fallen Earth,play,2.4


In [4]:
# Оставляем игры, у которых минимум 20 пользователей
df = df[df.groupby('game_name').user_id.transform('count') >= 20]
df['user_id'] = df['user_id'].astype(str)
df

,user_id,game_name,status,played_hour
1,151603712,The Elder Scrolls V Skyrim,play,273.0
3,151603712,Fallout 4,play,87.0
5,151603712,Spore,play,14.9
7,151603712,Fallout New Vegas,play,12.1
9,151603712,Left 4 Dead 2,play,8.9
...,...,...,...,...
199981,128470551,Hammerwatch,play,9.1
199983,128470551,Torchlight II,play,2.9
199987,128470551,Rogue Legacy,play,2.6
199989,128470551,Mortal Kombat Komplete Edition,play,2.5


In [5]:
# Среднее время игры по каждой игре
avg_time = df.groupby('game_name', as_index=False)['played_hour'].mean()
avg_time.rename(columns={'played_hour': 'avg_hourplayed'}, inplace=True)
df = df.merge(avg_time, on='game_name')
df

,user_id,game_name,status,played_hour,avg_hourplayed
0,151603712,The Elder Scrolls V Skyrim,play,273.0,115.351792
1,151603712,Fallout 4,play,87.0,66.819876
2,151603712,Spore,play,14.9,37.708889
3,151603712,Fallout New Vegas,play,12.1,62.910638
4,151603712,Left 4 Dead 2,play,8.9,50.333684
...,...,...,...,...,...
36540,128470551,Hammerwatch,play,9.1,14.244444
36541,128470551,Torchlight II,play,2.9,39.015341
36542,128470551,Rogue Legacy,play,2.6,18.191489
36543,128470551,Mortal Kombat Komplete Edition,play,2.5,28.962222


In [6]:
# Присваиваем рейтинг от 1 до 5 на основе доли от среднего времени
conditions = [
    df['played_hour'] >= 0.8 * df['avg_hourplayed'],
    df['played_hour'] >= 0.6 * df['avg_hourplayed'],
    df['played_hour'] >= 0.4 * df['avg_hourplayed'],
    df['played_hour'] >= 0.2 * df['avg_hourplayed'],
    df['played_hour'] >= 0
]
ratings = [5, 4, 3, 2, 1]
df['rating'] = np.select(conditions, ratings)

# Финальный датафрейм
df = df[['user_id', 'game_name', 'rating']]
df["user_id"] = df["user_id"].astype(np.int64)
df.head()


,user_id,game_name,rating
0,151603712,The Elder Scrolls V Skyrim,5
1,151603712,Fallout 4,5
2,151603712,Spore,2
3,151603712,Fallout New Vegas,1
4,151603712,Left 4 Dead 2,1


In [7]:
def recommend_similar_games(game, top_n=5):
    if game not in df_item_sim.columns:
        print(f'Игра "{game}" не найдена.')
        return
    print(f'Похожие игры на "{game}":')
    similar = df_item_sim[game].sort_values(ascending=False).iloc[1:top_n+1]
    for i, name in enumerate(similar.index, 1):
        print(f'{i}. {name}')


In [8]:
def similar_users(user_id, top_n=5):
    if user_id not in df_user_sim.columns:
        print(f'Пользователь {user_id} не найден.')
        return
    similar = df_user_sim[user_id].sort_values(ascending=False).iloc[1:top_n+1]
    for i, (uid, score) in enumerate(similar.items(), 1):
        print(f'{i}. User {uid}, similarity: {score:.2f}')


In [9]:
def recommend_from_similar_users(user_id, top_n=5):
    if user_id not in pivot.columns:
        print(f'Пользователь {user_id} не найден.')
        return []
    sim_users = df_user_sim[user_id].sort_values(ascending=False).index[1:11]
    game_counts = {}
    for u in sim_users:
        max_score = pivot[u].max()
        top_games = pivot[pivot[u] == max_score].index.tolist()
        for game in top_games:
            game_counts[game] = game_counts.get(game, 0) + 1
    sorted_games = sorted(game_counts.items(), key=operator.itemgetter(1), reverse=True)
    return [game for game, count in sorted_games[:top_n]]


In [16]:
from sklearn.model_selection import train_test_split

# Разбиваем на train/test
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
pv_train = train_df.pivot_table(index='user_id', columns='game_name', values='rating')
pv_train = pv_train.fillna(0).T

# Повторно считаем косинусное сходство
user_sim_eval = cosine_similarity(pv_train.T)
df_user_eval = pd.DataFrame(user_sim_eval, index=pv_train.columns, columns=pv_train.columns)

# Рекомендации по пользователю
def get_knn_recommendations(user_id, k=5):
    if user_id not in pv_train.columns:
        return []
    sim_users = df_user_eval[user_id].sort_values(ascending=False).index[1:11]
    recommended = []
    for u in sim_users:
        if u not in pv_train.columns:
            continue
        max_score = pv_train[u].max()
        recommended += pv_train[pv_train[u] == max_score].index.tolist()
    return list(dict.fromkeys(recommended))[:k]

# Метрики Precision@K и Recall@K
def evaluate(test_df, get_recommendations_func, k=5):
    users = test_df['user_id'].unique()
    total_precision = 0
    total_recall = 0
    total_f1 = 0
    hits_total = 0
    total_users = 0
    all_recommended_items = set()
    all_items = set(test_df['game_name'])

    for user in users:
        actual_items = test_df[test_df['user_id'] == user]['game_name'].tolist( )
        predicted_items = get_recommendations_func(user, k)

        if not predicted_items:
            continue

        hit_items = set(predicted_items) & set(actual_items)
        hits = len(hit_items)
        precision = hits / k
        recall = hits / len(actual_items) if actual_items else 0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0

        total_precision += precision
        total_recall += recall
        total_f1 += f1
        hits_total += hits
        total_users += 1
        all_recommended_items.update(predicted_items)

    if total_users == 0:
        print("No recommendations were made.")
        return

    print(f'📈 Precision@{k}: {total_precision / total_users:.2%}')
    print(f'📉 Recall@{k}: {total_recall / total_users:.2%}')
    print(f'🎯 F1@{k}: {(total_f1 / total_users):.2%}')
    print(f'🌍 Coverage@{k}: {len(all_recommended_items) / len(all_items):.2%}')


# Coverage
def coverage(df_user_eval, pv_train, k=5):
    all_recs = set()
    for user in pv_train.columns:
        recs = get_knn_recommendations(user, k=k)
        all_recs.update(recs)
    coverage_ratio = len(all_recs) / pv_train.shape[0]
    print(f'Coverage@{k}: {coverage_ratio:.2%}')

# Вызов метрик
evaluate(test_df, get_knn_recommendations, k=5)
coverage(df_user_eval, pv_train, k=5)


📈 Precision@5: 1.63%
📉 Recall@5: 3.87%
🎯 F1@5: 1.99%
🌍 Coverage@5: 92.99%
Coverage@5: 96.06%


In [17]:
# Матрица user-item
pivot = df.pivot_table(index='user_id', columns='game_name', values='rating')
pivot = pivot.apply(lambda x: (x - x.mean()) / (x.max() - x.min()), axis=1)
pivot = pivot.fillna(0).T  # item-user
pivot = pivot.loc[:, (pivot != 0).any(axis=0)]  # убираем пустые пользователи
df["user_id"] = df["user_id"].astype(np.int64)

# Разреженная матрица
sparse_matrix = sp.sparse.csr_matrix(pivot.values)


In [18]:
# Косинусное сходство
item_sim = cosine_similarity(sparse_matrix)
user_sim = cosine_similarity(sparse_matrix.T)

df_item_sim = pd.DataFrame(item_sim, index=pivot.index, columns=pivot.index)
df_user_sim = pd.DataFrame(user_sim, index=pivot.columns, columns=pivot.columns)

In [19]:
knn_model = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=6, n_jobs=-1)
knn_model.fit(pivot)

# Случайная игра и рекомендации
random_index = np.random.choice(pivot.shape[0])
chosen_game = pivot.index[random_index]
print(f"🎲 Случайно выбрана игра: {chosen_game}")

distances, indices = knn_model.kneighbors(pivot.iloc[random_index, :].values.reshape(1, -1))

for i in range(1, len(indices.flatten())):
    print(f"{i}. {pivot.index[indices.flatten()[i]]}, расстояние: {distances.flatten()[i]:.2f}")


🎲 Случайно выбрана игра: Sid Meier's Civilization V
1. Sid Meier's Civilization IV, расстояние: 0.91
2. Audiosurf, расстояние: 0.92
3. Warhammer 40,000 Space Marine, расстояние: 0.93
4. Don't Starve, расстояние: 0.93
5. Planetary Annihilation, расстояние: 0.93


In [20]:
df.dtypes

user_id       int64
game_name    object
rating        int32
dtype: object

In [21]:
# Импортируем необходимые библиотеки
from surprise import SVD, Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import accuracy


In [22]:
def get_svd_recommendations(data, user_id, k=5):
    if str(user_id) not in trainset._raw2inner_id_users:
        return [] 

    # Получаем список всех игр, которые пользователь еще не оценивал
    user_items = data[data['user_id'] == user_id]['game_name'].tolist()
    all_items = data['game_name'].unique()
    unseen_items = [item for item in all_items if item not in user_items]

    predictions = []
    for item in unseen_items:
        try:
            pred = algo.predict(str(user_id), item)
            predictions.append((item, pred.est))
        except:
            continue

    # Сортируем по предсказанному рейтингу и возвращаем top-k
    top_k_items = sorted(predictions, key=lambda x: x[1], reverse=True)[:k]
    return [item for item, _ in top_k_items]


In [24]:
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df, reader)

# Разделим на обучающую и тестовую выборку
trainset, testset = train_test_split(data, test_size=0.2)

# Обучаем модель SVD
svd = SVD()
svd.fit(trainset)

# Тестовый df (для evaluate)
test_df = pd.DataFrame(testset, columns=['user_id', 'game_name', 'rating'])

# Получаем предсказания
predictions = svd.test(testset)

# Оценка точности модели
accuracy.rmse(predictions);

evaluate(test_df, get_svd_recommendations, k=5)


RMSE: 1.5966
No recommendations were made.


In [ ]:
# #!/usr/bin/env python
# # coding: utf-8

# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt
# from sklearn.metrics.pairwise import cosine_similarity
# from sklearn.neighbors import NearestNeighbors
# from surprise import SVD, Dataset, Reader
# from surprise.model_selection import train_test_split as surprise_train_test_split
# from surprise import accuracy
# import warnings

# warnings.filterwarnings('ignore')

# # Загрузка и предварительная обработка данных
# df = pd.read_csv('source/steam_200k.csv', header=None)
# df.columns = ['user_id', 'game_name', 'status', 'hours_played', 'extra']  # Присваиваем имена колонкам
# df = df[['user_id', 'game_name', 'status', 'hours_played']]  # Удаляем лишнюю колонку
# df.drop_duplicates(inplace=True)  # Удаляем дубликаты

# # Фильтрация: оставляем только статус 'play' и время игры >= 2 часов
# df = df[(df['status'] == 'play') & (df['hours_played'] >= 2)]

# # Фильтруем игры, у которых минимум 20 уникальных пользователей
# game_counts = df.groupby('game_name')['user_id'].nunique()
# valid_games = game_counts[game_counts >= 20].index
# df = df[df['game_name'].isin(valid_games)]

# # Преобразуем user_id в строку для единообразия
# df['user_id'] = df['user_id'].astype(str)

# # Вычисляем среднее время игры для каждой игры и присваиваем рейтинги
# avg_time = df.groupby('game_name', as_index=False)['hours_played'].mean()
# avg_time.rename(columns={'hours_played': 'avg_hours_played'}, inplace=True)
# df = df.merge(avg_time, on='game_name')

# # Присваиваем рейтинги на основе доли от среднего времени игры
# conditions = [
#     (df['hours_played'] >= 0.8 * df['avg_hours_played']),
#     (df['hours_played'] >= 0.6 * df['avg_hours_played']),
#     (df['hours_played'] >= 0.4 * df['avg_hours_played']),
#     (df['hours_played'] >= 0.2 * df['avg_hours_played']),
#     (df['hours_played'] >= 0)
# ]
# ratings = [5, 4, 3, 2, 1]
# df['rating'] = np.select(conditions, ratings, default=1)  # Значение по умолчанию 1, если условие не выполнено

# # Финальный датафрейм для анализа
# df_final = df[['user_id', 'game_name', 'rating']]
# print("Размер финального датасета:", df_final.shape)
# print("Первые строки датасета:\n", df_final.head())

# # Бейзлайн: Рекомендательная система на основе KNN
# # Создаем матрицу user-item с нормализацией
# pivot_knn = df_final.pivot_table(index='user_id', columns='game_name', values='rating')
# pivot_knn = pivot_knn.apply(lambda x: (x - x.mean()) / (x.max() - x.min()) if x.max() > x.min() else x, axis=1)
# pivot_knn = pivot_knn.fillna(0).T  # Транспонируем в item-user
# pivot_knn = pivot_knn.loc[:, (pivot_knn != 0).any(axis=0)]  # Удаляем пользователей без рейтингов

# # Создаем разреженную матрицу для эффективности
# sparse_matrix = pivot_knn.values
# knn_model = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=11, n_jobs=-1)
# knn_model.fit(sparse_matrix)

# # Функция рекомендаций для KNN
# def knn_recommendations(game, top_n=5):
#     if game not in pivot_knn.index:
#         print(f'Игра "{game}" не найдена.')
#         return []
#     game_idx = pivot_knn.index.get_loc(game)
#     distances, indices = knn_model.kneighbors(sparse_matrix[game_idx].reshape(1, -1), n_neighbors=top_n + 1)
#     similar_games = [pivot_knn.index[i] for i in indices.flatten()[1:]]
#     return similar_games

# # Финальная модель: Рекомендательная система на основе SVD
# reader = Reader(rating_scale=(1, 5))
# data = Dataset.load_from_df(df_final[['user_id', 'game_name', 'rating']], reader)

# # Разделяем данные на обучающую и тестовую выборки с использованием surprise
# trainset, testset = surprise_train_test_split(data, test_size=0.2, random_state=42)

# # Обучаем модель SVD с настроенными гиперпараметрами
# svd = SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
# svd.fit(trainset)

# # Генерируем предсказания
# predictions = svd.test(testset)
# print(f"RMSE модели SVD: {accuracy.rmse(predictions):.4f}")

# # Улучшенная функция рекомендаций для SVD
# def svd_recommendations(user_id, top_n=5, model=svd, pivot=pivot_knn):
#     if user_id not in pivot.columns:
#         print(f'Пользователь {user_id} не найден.')
#         return []
#     # Получаем игры, которые пользователь уже оценил
#     rated_games = pivot[user_id][pivot[user_id] > 0].index
#     all_games = pivot.index
#     unrated_games = [g for g in all_games if g not in rated_games]
    
#     # Предсказываем рейтинги для неоцененных игр
#     predictions = [(game, svd.predict(user_id, game).est) for game in unrated_games]
#     predictions.sort(key=lambda x: x[1], reverse=True)
#     return [game for game, _ in predictions[:top_n]]

# # Функция оценки рекомендаций
# def evaluate_recommendations(test_df, model_func, k=5, model_name="SVD"):
#     users = test_df['user_id'].unique()
#     precision_sum, recall_sum, coverage_set = 0, 0, set()
#     n_users = 0

#     for user in users:
#         actual = set(test_df[test_df['user_id'] == user]['game_name'].tolist())
#         if not actual:
#             continue
#         predicted = set(model_func(user, k))
#         if not predicted:
#             continue
        
#         hits = actual & predicted
#         precision_sum += len(hits) / k
#         recall_sum += len(hits) / len(actual)
#         coverage_set.update(predicted)
#         n_users += 1

#     precision = precision_sum / n_users if n_users > 0 else 0
#     recall = recall_sum / n_users if n_users > 0 else 0
#     coverage = len(coverage_set) / len(df_final['game_name'].unique()) if df_final['game_name'].unique().size > 0 else 0

#     print(f"Метрики {model_name} @ {k}:")
#     print(f"Точность (Precision): {precision:.2%}")
#     print(f"Полнота (Recall): {recall:.2%}")
#     print(f"Покрытие (Coverage): {coverage:.2%}")
#     return precision, recall, coverage

# # Оценка бейзлайна (KNN)
# print("\nОценка бейзлайна KNN:")
# knn_precision, knn_recall, knn_coverage = evaluate_recommendations(test_df, knn_recommendations, k=5, model_name="KNN")

# # Оценка финальной модели (SVD)
# print("\nОценка финальной модели SVD:")
# svd_precision, svd_recall, svd_coverage = evaluate_recommendations(test_df, svd_recommendations, k=5, model_name="SVD")

# # Пример рекомендаций для случайного пользователя
# random_user = df_final['user_id'].sample().iloc[0]
# print(f"\nПример рекомендаций для пользователя {random_user}:")
# print("Рекомендации KNN:", knn_recommendations(random_user, top_n=5))
# print("Рекомендации SVD:", svd_recommendations(random_user, top_n=5))

In [ ]:
# #!/usr/bin/env python
# # coding: utf-8

# import pandas as pd
# import numpy as np
# from sklearn.neighbors import NearestNeighbors
# from surprise import SVD, Dataset, Reader
# from surprise.model_selection import train_test_split
# from surprise import accuracy
# import warnings

# warnings.filterwarnings('ignore')

# # ██████╗  ██████╗  ██████╗██████╗ ███████╗
# # ██╔══██╗██╔═══██╗██╔════╝██╔══██╗██╔════╝
# # ██║  ██║██║   ██║██║     ██████╔╝███████╗
# # ██║  ██║██║   ██║██║     ██╔═══╝ ╚════██║
# # ██████╔╝╚██████╔╝╚██████╗██║     ███████║
# # ╚═════╝  ╚═════╝  ╚═════╝╚═╝     ╚══════╝

# # =============================================
# # 1. ЗАГРУЗКА И ПРЕДОБРАБОТКА ДАННЫХ
# # =============================================

# # Загрузка сырых данных
# df = pd.read_csv('source/steam_200k.csv', header=None)
# df.columns = ['user_id', 'game_name', 'action', 'hours', 'unknown']
# df = df[['user_id', 'game_name', 'hours']]

# # Фильтрация данных
# df = df[df['hours'] >= 2]  # Игры с наигранным временем от 2 часов
# df = df.groupby('game_name').filter(lambda x: len(x) >= 20)  # Популярные игры

# # Присвоение рейтингов на основе времени игры
# df['rating'] = pd.qcut(df['hours'], q=5, labels=[1, 2, 3, 4, 5])
# df['rating'] = df['rating'].astype(np.int16)
# df = df[['user_id', 'game_name', 'rating']]

# # Уникальные идентификаторы
# print(f"Уникальных пользователей: {df['user_id'].nunique()}")
# print(f"Уникальных игр: {df['game_name'].nunique()}")

# # =============================================
# # 2. БЕЙЗЛАЙН: KNN МОДЕЛЬ
# # =============================================

# # Создание user-item матрицы
# user_item_matrix = df.pivot_table(
#     index='game_name',
#     columns='user_id',
#     values='rating',
#     fill_value=0
# )

# # Оптимизированный алгоритм ближайших соседей
# knn_model = NearestNeighbors(
#     metric='cosine', 
#     algorithm='brute', 
#     n_neighbors=11
# )
# knn_model.fit(user_item_matrix)

# def knn_recommend(game_name: str, n_recommendations: int = 5) -> list:
#     """
#     Рекомендации на основе схожести игр (item-based)
    
#     Параметры:
#     - game_name: Название исходной игры
#     - n_recommendations: Количество рекомендаций
    
#     Возвращает:
#     - Список рекомендованных игр
#     """
#     if game_name not in user_item_matrix.index:
#         return []
    
#     idx = user_item_matrix.index.get_loc(game_name)
#     distances, indices = knn_model.kneighbors(
#         user_item_matrix.iloc[idx:idx+1], 
#         n_neighbors=n_recommendations+1
#     )
    
#     return [
#         user_item_matrix.index[i] 
#         for i in indices.flatten()[1:]
#     ]

# # =============================================
# # 3. ФИНАЛЬНАЯ МОДЕЛЬ: SVD
# # =============================================

# # Подготовка данных для Surprise
# reader = Reader(rating_scale=(1, 5))
# data = Dataset.load_from_df(df, reader)

# # Разделение на train/test
# trainset, testset = train_test_split(data, test_size=0.2)

# # Обучение модели с оптимизированными параметрами
# svd_model = SVD(
#     n_factors=100,
#     n_epochs=20,
#     lr_all=0.005,
#     reg_all=0.02,
#     random_state=42
# )
# svd_model.fit(trainset)

# # Прогнозирование на тестовых данных
# predictions = svd_model.test(testset)
# print(f"\nОценка качества SVD:")
# print(f"RMSE: {accuracy.rmse(predictions):.4f}")

# def svd_recommend(user_id: str, n_recommendations: int = 5) -> list:
#     """
#     Персонализированные рекомендации с учетом скрытых факторов
    
#     Параметры:
#     - user_id: Идентификатор пользователя
#     - n_recommendations: Количество рекомендаций
    
#     Возвращает:
#     - Список рекомендованных игр
#     """
#     # Получение списка всех игр
#     all_games = df['game_name'].unique()
    
#     # Игры, которые пользователь уже оценил
#     rated_games = df[df['user_id'] == user_id]['game_name']
    
#     # Предсказание рейтингов для неоцененных игр
#     predictions = []
#     for game in all_games:
#         if game not in rated_games:
#             pred = svd_model.predict(user_id, game)
#             predictions.append((game, pred.est))
    
#     # Сортировка по убыванию рейтинга
#     predictions.sort(key=lambda x: x[1], reverse=True)
    
#     return [game for game, _ in predictions[:n_recommendations]]

# # =============================================
# # 4. ОЦЕНКА И СРАВНЕНИЕ МОДЕЛЕЙ
# # =============================================

# def evaluate_model(test_data, model_func, model_name: str):
#     """
#     Расчет метрик Precision@5 и Recall@5
    
#     Параметры:
#     - test_data: Тестовый датасет
#     - model_func: Функция для получения рекомендаций
#     - model_name: Название модели для вывода
#     """
#     precision_sum = 0
#     recall_sum = 0
#     users_count = 0
    
#     for user_id in test_data['user_id'].unique():
#         # Фактически сыгранные игры
#         actual = set(test_data[test_data['user_id'] == user_id]['game_name'])
#         if not actual:
#             continue
        
#         # Полученные рекомендации
#         recommended = set(model_func(user_id))
#         if not recommended:
#             continue
        
#         # Расчет метрик
#         hits = actual & recommended
#         precision_sum += len(hits) / 5
#         recall_sum += len(hits) / len(actual)
#         users_count += 1
    
#     print(f"\n{model_name} Metrics:")
#     print(f"Precision@5: {precision_sum / users_count:.2%}")
#     print(f"Recall@5: {recall_sum / users_count:.2%}")

# # Оценка бейзлайна
# evaluate_model(df, lambda uid: knn_recommend(uid), "KNN Baseline")

# # Оценка SVD
# evaluate_model(df, lambda uid: svd_recommend(uid), "SVD Final Model")

# # =============================================
# # 5. ПРИМЕРЫ РЕКОМЕНДАЦИЙ
# # =============================================

# # Пример для случайного пользователя
# sample_user = df['user_id'].sample().iloc[0]
# print(f"\nПример рекомендаций для пользователя {sample_user}:")
# print("KNN:", knn_recommend(sample_user))
# print("SVD:", svd_recommend(sample_user))

# # Пример для популярной игры
# sample_game = df['game_name'].value_counts().index[0]
# print(f"\nИгры, похожие на '{sample_game}':")
# print(knn_recommend(sample_game))